# ROC Curve

This notebook builds the ROC curve for the current saved checkpoint and exports a PNG for presentation use.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix, roc_curve, roc_auc_score

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.inference.resume_screener import ResumeScreeningModel
from ml.training.train_model import DEFAULT_THRESHOLD, load_processed_tensors

processed_dir = PROJECT_ROOT / "data" / "processed"
model_path = PROJECT_ROOT / "ml" / "models" / "resume_net.pt"
plots_dir = PROJECT_ROOT / "notebooks" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

_, X_test, _, y_test = load_processed_tensors(processed_dir=processed_dir)
model = ResumeScreeningModel(str(model_path), device="cpu", threshold=DEFAULT_THRESHOLD)

with torch.no_grad():
    probabilities = torch.sigmoid(model.model(X_test)).detach().cpu().numpy().reshape(-1)

labels = y_test.cpu().numpy().reshape(-1).astype(int)
predictions = (probabilities >= DEFAULT_THRESHOLD).astype(int)

print(f"Threshold: {DEFAULT_THRESHOLD}")
print(f"Test size: {labels.shape[0]}")


In [ ]:
fpr, tpr, _ = roc_curve(labels, probabilities)
auc_roc = roc_auc_score(labels, probabilities)
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc_roc, estimator_name="ResumeNet").plot(ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.set_title("ROC Curve for Adaptive Resume Screener")
fig.tight_layout()
output_path = plots_dir / "roc_curve.png"
fig.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"AUC-ROC: {auc_roc:.4f}")
print(f"Saved plot to: {output_path}")
